In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import time
from tqdm import tqdm
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D
from caveclient import CAVEclient
from nglui import parser
from nglui.statebuilder import helpers
datastack_name = 'minnie65_public'
client = CAVEclient(datastack_name)

# Show the description of the datastack
client.info.get_datastack_info()['description']

'This is the publicly released version of the minnie65 volume and segmentation. '

In [2]:
all_data = []
for file in glob.glob('cnn_unit_corrn_sess_*.csv'):
    df = pd.read_csv(file)
    parts = os.path.basename(file).split('_')
    session = int(parts[4])
    scan = int(parts[6].split('.')[0]) 
    df['session'] = session
    df['scan'] = scan
    all_data.append(df)
all_data = pd.concat(all_data).drop_duplicates().reset_index(drop=True)
rnn_all_data = []
for file in glob.glob('unit_corrn_sess_*.csv'):
    df = pd.read_csv(file)
    parts = os.path.basename(file).split('_')
    session = int(parts[3])
    scan = int(parts[5].split('.')[0])
    df['session'] = session
    df['scan'] = scan
    rnn_all_data.append(df)
rnn_all_data.append(df)
rnn_all_data = pd.concat(rnn_all_data).drop_duplicates().reset_index(drop=True).rename(columns={'corrn':'correlation'})
rnn_all_data

ValueError: No objects to concatenate

In [ ]:
session_stats = all_data.groupby('session')['correlation'].describe()
session_stats

In [ ]:
rnn_session_stats = rnn_all_data.groupby('session')['correlation'].describe()
rnn_session_stats

In [ ]:
plt.figure(figsize=(14, 8))
sns.boxplot(x='session', y='correlation', data=all_data)
sns.stripplot(x='session', y='correlation', data=all_data, size=1, alpha=0.3, color='black')
plt.title('Distribution of Correlation Values by Session', fontsize=16)
plt.xlabel('Session', fontsize=14)
plt.ylabel('Correlation', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.boxplot(x='session', y='correlation', data=rnn_all_data)
sns.stripplot(x='session', y='correlation', data=rnn_all_data, size=1, alpha=0.3, color='black')
plt.title('RNN Distribution of Correlation Values by Session', fontsize=16)
plt.xlabel('Session', fontsize=14)
plt.ylabel('Correlation', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(x='depth', y='correlation', hue='session', data=all_data, alpha=0.5, size=1)
plt.title('Correlation vs. Depth by Session', fontsize=16)
plt.xlabel('Depth', fontsize=14)
plt.ylabel('Correlation', fontsize=14)
plt.legend(title='Session')
plt.show()

In [ ]:
all_data['depth_bin'] = pd.cut(all_data['depth'], bins=20)
pivot_data = all_data.pivot_table(observed=False,values='correlation', index='depth_bin', columns='session', aggfunc='mean')

plt.figure(figsize=(14, 10))
sns.heatmap(pivot_data, cmap='viridis', annot=True)
plt.title('Average Correlation by Depth and Session', fontsize=18)
plt.xlabel('Session', fontsize=12)
plt.ylabel('Depth', fontsize=12)
plt.show()

In [ ]:
'''
unit_scan_counts = all_data.groupby(['session', 'unit_id'])['scan'].nunique().reset_index()
multi_scan_units = unit_scan_counts[unit_scan_counts['scan'] > 1]
unit_means = all_data.merge(multi_scan_units[['session', 'unit_id']], on=['session', 'unit_id']).groupby(['session', 'unit_id'])['correlation'].mean().reset_index()
top_units = unit_means.groupby('session').apply(lambda x: x.nlargest(10, 'correlation')).reset_index(drop=True)

top_units_table = top_units.pivot(index='unit_id', columns='session', values='correlation')
top_units_table = top_units_table.round(3)
'''

In [ ]:
output_dir = "microns_data_files"
os.makedirs(output_dir, exist_ok=True)
tables = [
    'baylor_gnn_cell_type_fine_model_v2',
    'nucleus_alternative_points',
    'allen_column_mtypes_v2',
    'bodor_pt_cells',
    'aibs_metamodel_mtypes_v661_v2',
    'allen_v1_column_types_slanted_ref',
    'aibs_column_nonneuronal_ref',
    'nucleus_ref_neuron_svm',
    'apl_functional_coreg_vess_fwd',
    'baylor_log_reg_cell_type_coarse_v1',
    'functional_properties_v3_bcm',
    'gamlin_2023_mcs',
    'l5et_column',
    'pt_synapse_targets',
    'coregistration_manual_v4',
    'cg_cell_type_calls',
    'synapses_pni_2',
    'nucleus_detection_v0',
    'vortex_manual_nodes_of_ranvier',
    'vortex_astrocyte_proofreading_status',
    'bodor_pt_target_proofread',
    'nucleus_functional_area_assignment',
    'coregistration_auto_phase3_fwd_apl_vess_combined_v2',
    'synapse_target_structure',
    'coregistration_auto_phase3_fwd_v2',
    'gamlin_2023_mcs_met_types',
    'vortex_manual_myelination_v0',
    'proofreading_status_and_strategy',
    'synapse_target_predictions_ssa',
    'aibs_metamodel_celltypes_v661'
]

os.makedirs("microns_data_files", exist_ok=True)
table_name = 'baylor_log_reg_cell_type_coarse_v1'
file_path = os.path.join("microns_data_files", table_name + ".csv")
df = client.materialize.query_table(table_name)
df.to_csv(file_path, index=False)


In [ ]:
aibs_df = client.materialize.query_table('aibs_metamodel_mtypes_v661_v2')
rnn_all_data = rnn_all_data[rnn_all_data['session'] == 6]

In [ ]:
functional_data_df = client.materialize.query_table('functional_properties_v3_bcm')[['pt_root_id', 'volume', 'unit_id', 'pref_ori', 'pref_dir', 'gOSI', 'gDSI', 'cc_abs']]

merged_data = pd.merge(all_data, functional_data_df, on='unit_id', how='inner')
rnn_merged_data = pd.merge(rnn_all_data, functional_data_df, on='unit_id', how='inner')
merged_stats = merged_data.describe()
merged_stats

In [ ]:
merged_data = merged_data.merge(right=client.materialize.query_table('aibs_metamodel_mtypes_v661_v2')[['pt_root_id', 'cell_type', 'classification_system']], left_on='pt_root_id', right_on='pt_root_id', how='inner')
rnn_merged_data = rnn_merged_data.merge(right=client.materialize.query_table('aibs_metamodel_mtypes_v661_v2')[['pt_root_id', 'cell_type', 'classification_system']], left_on='pt_root_id', right_on='pt_root_id', how='inner')

In [ ]:
plt.figure(figsize=(16, 10))
sns.boxplot(x='cell_type', y='correlation', data=merged_data)
plt.title('Correlation by Cell Type', fontsize=16)
plt.xlabel('Cell Type', fontsize=14)
plt.ylabel('Correlation', fontsize=14)
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(16, 10))
sns.boxplot(x='cell_type', y='correlation', data=rnn_merged_data)
plt.title('RNN Correlation by Cell Type', fontsize=16)
plt.xlabel('Cell Type', fontsize=14)
plt.ylabel('Correlation', fontsize=14)
plt.xticks(rotation=45)
plt.show()

In [ ]:
merged_data = merged_data[merged_data['session'] != 4]
rnn_merged_data = rnn_merged_data[rnn_merged_data['session'] != 4]

In [ ]:
corr_matrix = merged_data[['correlation', 'depth', 'volume', 'pref_ori', 'pref_dir', 'gOSI', 'gDSI']].corr()
rnn_corr_matrix = rnn_merged_data[['correlation', 'volume', 'pref_ori', 'pref_dir', 'gOSI', 'gDSI']].corr()

In [ ]:
plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, vmax=.3, linewidths=.5, annot=True)
plt.title('Correlation Matrix of Neural Properties', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
properties = ['correlation', 'volume', 'pref_ori', 'pref_dir', 'gOSI', 'gDSI']
p_matrix = np.zeros((7,7))
for i, prop1 in enumerate(properties):
    for j, prop2 in enumerate(properties):
        if i != j:
            corr, p = stats.pearsonr(merged_data[prop1], merged_data[prop2])
            p_matrix[i,j] = p
plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(p_matrix, dtype=bool))
sns.heatmap(p_matrix, mask=mask, vmax=.3, linewidths=.5, annot=True)      
plt.title('P-Value Matrix of Neural Properties', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(rnn_corr_matrix, dtype=bool))
sns.heatmap(rnn_corr_matrix, mask=mask, vmax=.3, linewidths=.5, annot=True)
plt.title('RNN Correlation Matrix of Neural Properties', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
properties = ['correlation', 'volume', 'pref_ori', 'pref_dir', 'gOSI', 'gDSI']
p_matrix = np.zeros((7,7))
for i, prop1 in enumerate(properties):
    for j, prop2 in enumerate(properties):
        if i != j:
            corr, p = stats.pearsonr(rnn_merged_data[prop1], rnn_merged_data[prop2])
            p_matrix[i,j] = p
plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(p_matrix, dtype=bool))
sns.heatmap(p_matrix, mask=mask, vmax=.3, linewidths=.5, annot=True)      
plt.title('RNN P-Value Matrix of Neural Properties', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(data=merged_data, x='correlation', y='pref_ori', alpha=0.6)
plt.title('Correlation vs Preferred Orientation', fontsize=16)
plt.xlabel('Preferred Orientation (pref_ori)', fontsize=14)
plt.ylabel('Correlation', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(data=rnn_merged_data, x='correlation', y='pref_ori', alpha=0.6)
plt.title('RNN Correlation vs Preferred Orientation', fontsize=16)
plt.xlabel('Preferred Orientation (pref_ori)', fontsize=14)
plt.ylabel('Correlation', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(x='gDSI', y='correlation', hue='session', data=merged_data, alpha=0.6)
plt.title('Correlation vs Direction Selectivity Index', fontsize=16)
plt.xlabel('Direction Selectivity Index (gDSI)', fontsize=14)
plt.ylabel('RNN Correlation', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(x='gDSI', y='correlation', data=rnn_merged_data, alpha=0.6)
plt.title('RNN Correlation vs Direction Selectivity Index', fontsize=16)
plt.xlabel('Direction Selectivity Index (gDSI)', fontsize=14)
plt.ylabel('RNN Correlation', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(x='gOSI', y='correlation', data=merged_data, alpha=0.6)
plt.title('Correlation vs Orientation Selectivity Index', fontsize=16)
plt.xlabel('Orientation Selectivity Index (gOSI)', fontsize=14)
plt.ylabel('RNN Correlation', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.regplot(x='gOSI', y='correlation', data=rnn_merged_data)
plt.title('RNN Correlation vs Orientation Selectivity Index', fontsize=16)
plt.xlabel('Orientation Selectivity Index (gOSI)', fontsize=14)
plt.ylabel('RNN Correlation', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(x='volume', y='correlation', data=merged_data)
plt.title('Correlation vs Volume', fontsize=16)
plt.xlabel('Volume', fontsize=14)
plt.ylabel('RNN Correlation', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.regplot(x='volume', y='correlation', data=rnn_merged_data)
plt.title('RNN Correlation vs Volume', fontsize=16)
plt.xlabel('Volume', fontsize=14)
plt.ylabel('RNN Correlation', fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(x='volume', y='correlation', hue='session', data=merged_data, alpha=0.6)
plt.title('Correlation vs Neuron Volume', fontsize=16)
plt.xlabel('Volume', fontsize=14)
plt.ylabel('Correlation', fontsize=14)
plt.legend(title='Session')
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(x='pref_dir', y='correlation', hue='session', data=merged_data, alpha=0.6)
plt.xlabel('Correlation')
plt.ylabel('prefdir')
plt.title("Correlation vs Preferred Direction")
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(x='pref_dir', y='correlation', data=rnn_merged_data, alpha=0.6)
plt.xlabel('Correlation')
plt.ylabel('prefdir')
plt.title("RNN Correlation vs Preferred Direction")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
rnn_merged_data

In [ ]:
kde_plot = sns.jointplot(data=rnn_merged_data, x="correlation", y="gOSI", kind="scatter",joint_kws={"s": 17, "alpha": 0.5},height=8)
kde_plot.plot_joint(sns.kdeplot, levels=6)
kde_plot.ax_joint.annotate(f"r = {stats.pearsonr(rnn_merged_data['correlation'], rnn_merged_data['gOSI'])[0]:.3f}",xy=(0.1, 0.9), xycoords='axes fraction')
plt.xlabel('Correlation')
plt.ylabel('gOSI (Orientation Selectivity Index)')
plt.show()

In [ ]:
kde_plot = sns.jointplot(data=rnn_merged_data, x="correlation", y="volume", kind="scatter",joint_kws={"s": 17, "alpha": 0.5},height=8)
kde_plot.plot_joint(sns.kdeplot, levels=6)
kde_plot.ax_joint.annotate(f"r = {stats.pearsonr(rnn_merged_data['correlation'], rnn_merged_data['volume'])[0]:.3f}",xy=(0.1, 0.9), xycoords='axes fraction')
plt.xlabel('Correlation')
plt.ylabel('Neuronal Volume')
plt.show()

In [ ]:
kde_plot = sns.jointplot(data=rnn_merged_data, x="correlation", y="pref_ori", kind="scatter",joint_kws={"s": 17, "alpha": 0.5},height=8)
kde_plot.plot_joint(sns.kdeplot, levels=6)
kde_plot.ax_joint.annotate(f"r = {stats.pearsonr(rnn_merged_data['correlation'], rnn_merged_data['pref_ori'])[0]:.3f}",xy=(0.1, 0.9), xycoords='axes fraction')
plt.xlabel('Correlation')
plt.ylabel('Preferred Orientation')
plt.show()

In [ ]:
features = rnn_merged_data[['correlation', 'pref_ori']].values
n_clusters = 4

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
    
kmeans = KMeans(n_clusters, random_state=42)
k_means_rnn_merged_data = rnn_merged_data.copy()
k_means_rnn_merged_data['cluster'] = kmeans.fit_predict(scaled_features)
centers = scaler.inverse_transform(kmeans.cluster_centers_)
    
for i in range(n_clusters):
    cluster_data = k_means_rnn_merged_data[k_means_rnn_merged_data['cluster'] == i]
    plt.scatter(cluster_data['correlation'], cluster_data['pref_ori'], label=f'Cluster {i+1}', alpha=0.7, s=50)
plt.scatter(centers[:, 0], centers[:, 1], c='black', marker='X', s=200, label='Cluster centers')
    
plt.xlabel('Correlation')
plt.ylabel('Preferred Orientation')
plt.title(f'K-means clustering of RNN Correlation vs. Preferred Orientation (k={n_clusters})')

In [ ]:
features = rnn_merged_data[['correlation', 'volume']].values
n_clusters = 4

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
    
kmeans = KMeans(n_clusters, random_state=42)
k_means_rnn_merged_data = rnn_merged_data.copy()
k_means_rnn_merged_data['cluster'] = kmeans.fit_predict(scaled_features)
centers = scaler.inverse_transform(kmeans.cluster_centers_)
    
for i in range(n_clusters):
    cluster_data = k_means_rnn_merged_data[k_means_rnn_merged_data['cluster'] == i]
    plt.scatter(cluster_data['correlation'], cluster_data['volume'], label=f'Cluster {i+1}', alpha=0.7, s=50)
plt.scatter(centers[:, 0], centers[:, 1], c='black', marker='X', s=200, label='Cluster centers')
    
plt.xlabel('Correlation')
plt.ylabel('Volume')
plt.title(f'K-means clustering of RNN Correlation vs. Volume (k={n_clusters})')

In [ ]:
features = rnn_merged_data[['correlation', 'gOSI']].values
n_clusters = 4

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
    
kmeans = KMeans(n_clusters, random_state=42)
k_means_rnn_merged_data = rnn_merged_data.copy()
k_means_rnn_merged_data['cluster'] = kmeans.fit_predict(scaled_features)
centers = scaler.inverse_transform(kmeans.cluster_centers_)
    
for i in range(n_clusters):
    cluster_data = k_means_rnn_merged_data[k_means_rnn_merged_data['cluster'] == i]
    plt.scatter(cluster_data['correlation'], cluster_data['gOSI'], label=f'Cluster {i+1}', alpha=0.7, s=50)
plt.scatter(centers[:, 0], centers[:, 1], c='black', marker='X', s=200, label='Cluster centers')
    
plt.xlabel('Correlation')
plt.ylabel('gOSI (Orientation Selectivity Index)')
plt.title(f'K-means clustering of RNN Correlation vs. gOSI (k={n_clusters})')

In [ ]:
def analyze_heterogeneity():
    """Analyze heterogeneity in neural responses and correlations."""
    # Load data
    data = rnn_merged_data.rename(columns={'correlation':'corrn'})
    
    # Create figure for heterogeneity analysis
    fig = plt.figure(figsize=(20, 16))
    fig.suptitle('Heterogeneity Analysis of Neural Correlations', fontsize=20, y=0.98)
    
    # Create 2x2 grid
    gs = plt.GridSpec(2, 2, figure=fig)
    
    # 1. K-means clustering with different features
    ax1 = fig.add_subplot(gs[0, 0])
    
    # Select features for clustering
    cluster_features = ['corrn', 'gOSI', 'volume']
    cluster_data = data[cluster_features].dropna()
    
    # Standardize features
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(cluster_data)
    
    # Determine optimal number of clusters using silhouette score
    from sklearn.metrics import silhouette_score
    silhouette_scores = []
    k_range = range(2, 8)
    
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(scaled_data)
        silhouette_scores.append(silhouette_score(scaled_data, labels))
    
    # Plot silhouette scores
    ax1.plot(k_range, silhouette_scores, 'o-')
    ax1.set_xlabel('Number of clusters')
    ax1.set_ylabel('Silhouette score')
    ax1.set_title('Optimal number of clusters')
    ax1.grid(True, alpha=0.3)
    
    # Find optimal k
    optimal_k = k_range[np.argmax(silhouette_scores)]
    ax1.axvline(optimal_k, color='red', linestyle='--')
    ax1.text(optimal_k + 0.1, min(silhouette_scores), f'Optimal k = {optimal_k}', 
             color='red', va='bottom')
    
    # 2. Visualize clusters in 3D space
    ax2 = fig.add_subplot(gs[0, 1], projection='3d')
    
    # Apply K-means with optimal k
    kmeans = KMeans(n_clusters=optimal_k, random_state=42)
    cluster_data['cluster'] = kmeans.fit_predict(scaled_data)
    
    # Get cluster centers
    centers = scaler.inverse_transform(kmeans.cluster_centers_)
    
    # Create 3D scatter plot
    for i in range(optimal_k):
        cluster_subset = cluster_data[cluster_data['cluster'] == i]
        ax2.scatter(
            cluster_subset['corrn'], 
            cluster_subset['gOSI'], 
            cluster_subset['volume'],
            label=f'Cluster {i+1}',
            alpha=0.7
        )
    
    # Plot cluster centers
    ax2.scatter(
        centers[:, 0], 
        centers[:, 1], 
        centers[:, 2],
        c='black',
        marker='X',
        s=200,
        label='Cluster centers'
    )
    
    ax2.set_xlabel('corrn')
    ax2.set_ylabel('gOSI')
    ax2.set_zlabel('volume')
    ax2.set_title('3D visualization of neuron clusters')
    ax2.legend()
    
    # 3. Parallel coordinates plot to visualize clusters
    ax3 = fig.add_subplot(gs[1, 0])
    
    # Add cluster column to original data
    cluster_column = pd.Series(index=cluster_data.index, data=cluster_data['cluster'].values)
    plot_data = data.loc[cluster_data.index].copy()
    plot_data['cluster'] = cluster_column
    
    # Create parallel coordinates plot
    from pandas.plotting import parallel_coordinates
    
    # Normalize data for better visualization
    plot_features = ['corrn', 'gOSI', 'volume', 'pref_ori']
    plot_data_norm = plot_data[plot_features].copy()
    for feature in plot_features:
        plot_data_norm[feature] = (plot_data_norm[feature] - plot_data_norm[feature].min()) / \
                                 (plot_data_norm[feature].max() - plot_data_norm[feature].min())
    plot_data_norm['cluster'] = plot_data['cluster']
    
    # Plot
    parallel_coordinates(plot_data_norm, 'cluster', ax=ax3, colormap='viridis')
    ax3.set_title('Parallel coordinates plot of neuron clusters')
    ax3.grid(True, alpha=0.3)
    
    # 4. Distribution of correlation values within each cluster
    ax4 = fig.add_subplot(gs[1, 1])
    
    # Create box plots
    sns.boxplot(x='cluster', y='corrn', data=plot_data, ax=ax4, palette='viridis')
    
    # Add swarm plots
    sns.swarmplot(x='cluster', y='corrn', data=plot_data, ax=ax4, color='black', alpha=0.6, size=4)
    
    ax4.set_title('Distribution of correlation values across clusters')
    ax4.set_xlabel('Cluster Number')
    ax4.set_ylabel('RNN Correlation')
    
    plt.tight_layout()    
    
    # Print cluster statistics
    print("\n=== Cluster Statistics ===")
    cluster_stats = plot_data.groupby('cluster').agg({
        'corrn': ['mean', 'std', 'count'],
        'gOSI': 'mean',
        'volume': 'mean',
        'pref_ori': lambda x: stats.circmean(x, high=np.pi, low=0)
    })
    
    print(cluster_stats)
    
    # Calculate ANOVA to test if clusters are significantly different
    from scipy.stats import f_oneway
    
    cluster_groups = [plot_data[plot_data['cluster'] == i]['corrn'].values 
                     for i in range(optimal_k)]
    f_val, p_val = f_oneway(*cluster_groups)
    
    print(f"\nANOVA test for difference in correlation across clusters:")
    print(f"F-value: {f_val:.3f}, p-value: {p_val:.3e}")
    
    if p_val < 0.05:
        print("The clusters show significantly different correlation values.")
    else:
        print("No significant difference in correlation values across clusters.")
    
    # Return data for further analysis
    return plot_data, optimal_k
analyze_heterogeneity()

In [ ]:
def comprehensive_neural_analysis(
    unit_corrn_pattern='unit_corrn_sess_*_scan_*.csv',
    cnn_unit_corrn_pattern='cnn_unit_corrn_sess_*_scan_*.csv',
    func_properties_file='functional_properties_v3_bcm.csv',
    output_dir='neural_analysis_results',
    dpi=300,
    seed=42
):
    """
    Perform comprehensive analysis and visualization of neural correlation data.
    
    This function integrates multiple visualization techniques and statistical analyses
    to explore relationships between RNN correlation values and functional properties,
    such as volume, preferred orientation, and orientation selectivity (gOSI).
    
    Parameters:
    -----------
    unit_corrn_pattern : str
        Glob pattern for unit correlation files
    cnn_unit_corrn_pattern : str
        Glob pattern for CNN unit correlation files
    func_properties_file : str
        Path to functional properties file
    output_dir : str
        Directory to save visualization outputs
    dpi : int
        Resolution for saved figures
    seed : int
        Random seed for reproducibility
    
    Returns:
    --------
    dict
        Dictionary containing analysis results and summary statistics
    """
    
    # Set random seed for reproducibility
    np.random.seed(seed)
    
    # Create output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Configure plotting style
    sns.set_context("paper", font_scale=1.5)
    plt.rcParams['axes.labelsize'] = 14
    plt.rcParams['axes.titlesize'] = 16
    plt.rcParams['xtick.labelsize'] = 12
    plt.rcParams['ytick.labelsize'] = 12
    
    # Create custom color palettes
    colors = sns.color_palette("viridis", 8)
    session_colors = sns.color_palette("Set2", 8)
    scan_colors = sns.color_palette("Set1", 8)
    
    # Helper function for log messages
    def log(message):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{timestamp}] {message}")
    
    # 1. Data Loading and Preparation
    # ===============================
    log("Starting comprehensive neural data analysis...")
    log("Loading and preparing data...")
    
    # Load unit correlation files
    unit_corrn_files = sorted(glob.glob(unit_corrn_pattern))
    log(f"Found {len(unit_corrn_files)} unit correlation files")
    
    # Load and merge correlation data
    unit_corrn_dfs = []
    for file_path in tqdm(unit_corrn_files, desc="Loading unit correlation files"):
        # Extract session and scan from filename
        parts = os.path.basename(file_path).split('_')
        session = int(parts[3])
        scan = int(parts[5].split('.')[0])
        
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Add session and scan information
        df['session'] = session
        df['scan'] = scan
        df['file_source'] = os.path.basename(file_path)
        
        unit_corrn_dfs.append(df)
    
    # Combine all correlation DataFrames
    if unit_corrn_dfs:
        unit_corrn_df = pd.concat(unit_corrn_dfs, ignore_index=True)
        log(f"Combined correlation data: {len(unit_corrn_df)} records")
    else:
        log("No unit correlation files found")
        return None
    
    # Load functional properties
    try:
        log(f"Loading functional properties from {func_properties_file}")
        func_df = pd.read_csv(func_properties_file)
        log(f"Loaded functional properties: {len(func_df)} records")
    except Exception as e:
        log(f"Error loading functional properties: {e}")
        return None
    
    # Rename scan_idx to scan for consistency
    if 'scan_idx' in func_df.columns:
        func_df = func_df.rename(columns={'scan_idx': 'scan'})
    
    # Merge correlation data with functional properties
    log("Merging correlation data with functional properties")
    merged_df = pd.merge(
        unit_corrn_df,
        func_df[['unit_id', 'session', 'scan', 'volume', 'pref_ori', 'gOSI', 'gDSI', 'cc_abs']],
        on=['unit_id', 'session', 'scan'],
        how='inner'
    )
    
    # Check if merge was successful
    if len(merged_df) == 0:
        log("Error: No matching records found after merging")
        return None
    
    log(f"Merged dataset: {len(merged_df)} records")
    
    # Clean data - remove any rows with NaN in critical columns
    critical_cols = ['corrn', 'volume', 'pref_ori', 'gOSI']
    clean_df = merged_df.dropna(subset=critical_cols)
    log(f"Clean dataset after removing NaNs: {len(clean_df)} records")
    
    # Normalize orientation to 0-π range (orientation has π periodicity)
    clean_df['pref_ori_norm'] = clean_df['pref_ori'] % np.pi
    
    # 2. Basic Descriptive Statistics
    # ===============================
    log("Calculating descriptive statistics...")
    
    # Overall statistics
    overall_stats = clean_df[['corrn', 'volume', 'pref_ori', 'gOSI']].describe()
    
    # Correlation matrix
    correlation_matrix = clean_df[['corrn', 'volume', 'pref_ori_norm', 'gOSI', 'gDSI', 'cc_abs']].corr()
    
    # Statistics by session
    session_stats = clean_df.groupby('session').agg({
        'corrn': ['count', 'mean', 'std', 'min', 'max'],
        'volume': 'mean',
        'gOSI': 'mean'
    })
    
    # Statistics by scan
    scan_stats = clean_df.groupby(['session', 'scan']).agg({
        'corrn': ['count', 'mean', 'std'],
        'gOSI': 'mean',
        'volume': 'mean'
    })
    
    # Calculate correlations for each session
    session_correlations = {}
    for session in clean_df['session'].unique():
        session_data = clean_df[clean_df['session'] == session]
        
        corr_gOSI, p_gOSI = pearsonr(session_data['corrn'], session_data['gOSI'])
        corr_vol, p_vol = pearsonr(session_data['corrn'], session_data['volume'])
        
        session_correlations[session] = {
            'count': len(session_data),
            'corrn_gOSI': corr_gOSI,
            'p_gOSI': p_gOSI,
            'corrn_volume': corr_vol,
            'p_volume': p_vol
        }
    
    # 3. Core Visualizations and Analysis
    # ===================================
    log("Generating core visualizations...")
    
    # 3.1 Correlation Analysis and Regression Plots
    # ---------------------------------------------
    
    # Function for regression plots with confidence intervals
    def create_regression_plot(df, x_var, y_var, title, filename, color_var=None):
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # Add scatter plot
        if color_var:
            scatter = ax.scatter(
                df[x_var], 
                df[y_var], 
                c=df[color_var], 
                cmap='viridis', 
                alpha=0.6, 
                s=40, 
                edgecolor='none'
            )
            plt.colorbar(scatter, ax=ax, label=color_var)
        else:
            ax.scatter(df[x_var], df[y_var], alpha=0.6, s=40, color=colors[0])
        
        # Calculate regression line
        slope, intercept, r_value, p_value, std_err = stats.linregress(df[x_var], df[y_var])
        
        # Add regression line
        x_range = np.linspace(df[x_var].min(), df[x_var].max(), 100)
        y_pred = intercept + slope * x_range
        ax.plot(x_range, y_pred, color='red', linewidth=2)
        
        # Add confidence interval
        from statsmodels.sandbox.regression.predstd import wls_prediction_std
        import statsmodels.api as sm
        
        X = sm.add_constant(df[x_var])
        model = sm.OLS(df[y_var], X).fit()
        
        # Calculate confidence intervals
        _, lower, upper = wls_prediction_std(model, sm.add_constant(x_range))
        
        # Plot confidence intervals
        ax.fill_between(x_range, lower, upper, color='red', alpha=0.2)
        
        # Add statistical information
        stat_text = (f"Slope: {slope:.3f}\n"
                     f"Intercept: {intercept:.3f}\n"
                     f"R²: {r_value**2:.3f}\n"
                     f"p-value: {p_value:.3e}")
        
        if p_value < 0.05:
            stat_text += "\nSignificant correlation"
        
        bbox_props = dict(boxstyle="round,pad=0.5", fc="white", ec="gray", alpha=0.8)
        ax.text(0.05, 0.95, stat_text, transform=ax.transAxes, 
                verticalalignment='top', bbox=bbox_props, fontsize=12)
        
        ax.set_xlabel(x_var)
        ax.set_ylabel(y_var)
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi)
        plt.close()
        
        return {
            'slope': slope,
            'intercept': intercept,
            'r_squared': r_value**2,
            'p_value': p_value,
            'std_err': std_err
        }
    
    # Create regression plots for key relationships
    reg_corrn_gOSI = create_regression_plot(
        clean_df, 'corrn', 'gOSI', 
        'RNN Correlation vs Orientation Selectivity',
        'regression_corrn_gOSI.png'
    )
    
    reg_corrn_volume = create_regression_plot(
        clean_df, 'corrn', 'volume', 
        'RNN Correlation vs Neuron Volume',
        'regression_corrn_volume.png'
    )
    
    # Regression with preferred orientation
    reg_corrn_ori = create_regression_plot(
        clean_df, 'corrn', 'pref_ori_norm', 
        'RNN Correlation vs Preferred Orientation',
        'regression_corrn_orientation.png'
    )
    
    # 3.2 Session-specific Correlation Analysis
    # ----------------------------------------
    log("Analyzing session-specific correlations...")
    
    # Plot correlation by session
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Correlation with gOSI by session
    gOSI_corrs = []
    gOSI_ps = []
    sessions = []
    
    for session in sorted(clean_df['session'].unique()):
        session_data = clean_df[clean_df['session'] == session]
        corr, p = pearsonr(session_data['corrn'], session_data['gOSI'])
        gOSI_corrs.append(corr)
        gOSI_ps.append(p)
        sessions.append(f"Session {session}")
    
    # Plot correlation coefficients
    bars = axes[0].bar(sessions, gOSI_corrs, color=[session_colors[i % len(session_colors)] for i in range(len(sessions))])
    
    # Add significance stars
    for i, p in enumerate(gOSI_ps):
        if p < 0.001:
            axes[0].text(i, gOSI_corrs[i] + 0.05, '***', ha='center')
        elif p < 0.01:
            axes[0].text(i, gOSI_corrs[i] + 0.05, '**', ha='center')
        elif p < 0.05:
            axes[0].text(i, gOSI_corrs[i] + 0.05, '*', ha='center')
    
    axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[0].set_ylabel('Correlation Coefficient (r)')
    axes[0].set_title('Correlation between RNN Correlation and gOSI by Session')
    axes[0].set_ylim(-0.5, 0.5)
    
    # Correlation with Volume by session
    volume_corrs = []
    volume_ps = []
    
    for session in sorted(clean_df['session'].unique()):
        session_data = clean_df[clean_df['session'] == session]
        corr, p = pearsonr(session_data['corrn'], session_data['volume'])
        volume_corrs.append(corr)
        volume_ps.append(p)
    
    # Plot correlation coefficients
    bars = axes[1].bar(sessions, volume_corrs, color=[session_colors[i % len(session_colors)] for i in range(len(sessions))])
    
    # Add significance stars
    for i, p in enumerate(volume_ps):
        if p < 0.001:
            axes[1].text(i, volume_corrs[i] + 0.05, '***', ha='center')
        elif p < 0.01:
            axes[1].text(i, volume_corrs[i] + 0.05, '**', ha='center')
        elif p < 0.05:
            axes[1].text(i, volume_corrs[i] + 0.05, '*', ha='center')
    
    axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[1].set_ylabel('Correlation Coefficient (r)')
    axes[1].set_title('Correlation between RNN Correlation and Volume by Session')
    axes[1].set_ylim(-0.5, 0.5)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'session_correlations.png'), dpi=dpi)
    plt.close()
    
    # 3.3 Binned Analysis and Trend Visualization
    # ------------------------------------------
    log("Performing binned analysis for trend visualization...")
    
    def create_binned_analysis_plot(df, x_var, y_var, n_bins=10, filename=None):
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Create bins
        bins = np.linspace(df[x_var].min(), df[x_var].max(), n_bins + 1)
        
        # Create bin labels
        bin_centers = [(bins[i] + bins[i+1]) / 2 for i in range(len(bins) - 1)]
        
        # Initialize arrays for means and errors
        means = []
        errors = []
        counts = []
        
        # Calculate statistics for each bin
        for i in range(len(bins) - 1):
            # Get data in this bin
            mask = (df[x_var] >= bins[i]) & (df[x_var] < bins[i+1])
            bin_data = df[mask][y_var]
            
            # Store statistics
            means.append(bin_data.mean() if len(bin_data) > 0 else np.nan)
            errors.append(bin_data.std() / np.sqrt(len(bin_data)) if len(bin_data) > 1 else np.nan)
            counts.append(len(bin_data))
        
        # Plot raw data
        ax.scatter(df[x_var], df[y_var], alpha=0.3, s=20, color='gray')
        
        # Plot binned means with error bars
        ax.errorbar(bin_centers, means, yerr=errors, fmt='o-', color=colors[0], 
                   linewidth=2, markersize=8, capsize=5, label='Binned means ± SEM')
        
        # Add count labels
        for i, (x, y, count) in enumerate(zip(bin_centers, means, counts)):
            ax.annotate(f'n={count}', (x, y), xytext=(0, 10), textcoords='offset points',
                       ha='center', fontsize=9)
        
        # Add trend line
        if all(not np.isnan(m) for m in means):
            from scipy.signal import savgol_filter
            if len(means) > 5:  # Only use Savitzky-Golay filter if we have enough points
                smooth_means = savgol_filter(means, min(5, len(means) - (len(means) % 2 - 1)), 3)
                ax.plot(bin_centers, smooth_means, '--', color='red', linewidth=2, label='Smoothed trend')
        
        ax.set_xlabel(x_var)
        ax.set_ylabel(f'Mean {y_var}')
        ax.set_title(f'Binned analysis of {y_var} by {x_var}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        if filename:
            plt.savefig(os.path.join(output_dir, filename), dpi=dpi)
            plt.close()
        
        return {
            'bin_centers': bin_centers,
            'means': means,
            'errors': errors,
            'counts': counts
        }
    
    # Binned analysis for key relationships
    binned_corrn_gOSI = create_binned_analysis_plot(
        clean_df, 'corrn', 'gOSI', n_bins=8, 
        filename='binned_corrn_gOSI.png'
    )
    
    binned_gOSI_corrn = create_binned_analysis_plot(
        clean_df, 'gOSI', 'corrn', n_bins=8, 
        filename='binned_gOSI_corrn.png'
    )
    
    binned_volume_corrn = create_binned_analysis_plot(
        clean_df, 'volume', 'corrn', n_bins=8, 
        filename='binned_volume_corrn.png'
    )
    
    # 3.4 Orientation-specific Analysis
    # --------------------------------
    log("Analyzing orientation preferences...")
    
    # Create polar plot of correlation by orientation
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='polar')
    
    # Create orientation bins (in radians)
    orientation_bins = np.linspace(0, np.pi, 13)  # 12 bins spanning 0 to π
    bin_centers = [(orientation_bins[i] + orientation_bins[i+1]) / 2 for i in range(len(orientation_bins) - 1)]
    
    # Calculate mean correlation for each orientation bin
    ori_means = []
    ori_errors = []
    ori_counts = []
    
    for i in range(len(orientation_bins) - 1):
        # Get data in this bin
        mask = (clean_df['pref_ori_norm'] >= orientation_bins[i]) & (clean_df['pref_ori_norm'] < orientation_bins[i+1])
        bin_data = clean_df[mask]['corrn']
        
        # Store statistics
        ori_means.append(bin_data.mean() if len(bin_data) > 0 else np.nan)
        ori_errors.append(bin_data.std() / np.sqrt(len(bin_data)) if len(bin_data) > 1 else np.nan)
        ori_counts.append(len(bin_data))
    
    # Plot binned means on polar axis
    bars = ax.bar(
        bin_centers, 
        ori_means, 
        width=np.pi/12, 
        alpha=0.7,
        color=plt.cm.viridis(np.linspace(0, 1, len(bin_centers)))
    )
    
    # Add count labels
    for i, (angle, radius, count) in enumerate(zip(bin_centers, ori_means, ori_counts)):
        if not np.isnan(radius):
            ax.text(
                angle, 
                radius + 0.05, 
                f'n={count}', 
                ha='center', 
                va='center', 
                rotation=np.degrees(angle) - 90,
                rotation_mode='anchor'
            )
    
    # Set axis limits and labels
    ax.set_theta_zero_location('N')  # 0 degrees at the top
    ax.set_theta_direction(-1)  # Clockwise
    ax.set_rlabel_position(0)  # Move radial labels to 0 degrees
    
    # Set custom theta labels (in degrees)
    ax.set_xticks(np.linspace(0, np.pi, 7))
    ax.set_xticklabels(['0°', '30°', '60°', '90°', '120°', '150°', '180°'])
    
    ax.set_title('RNN Correlation by Preferred Orientation', pad=20)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'orientation_polar_plot.png'), dpi=dpi)
    plt.close()
    
    # 3.5 Heatmap Visualizations
    # -------------------------
    log("Creating heatmap visualizations...")
    
    # 2D density heatmap of correlation vs gOSI
    def create_2d_density_heatmap(df, x_var, y_var, n_bins=20, filename=None):
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # Create 2D histogram
        h, x_edges, y_edges = np.histogram2d(
            df[x_var], 
            df[y_var], 
            bins=n_bins,
            density=True
        )
        
        # Convert to density
        h = h.T  # Transpose for correct orientation
        
        # Plot heatmap
        im = ax.imshow(
            h, 
            origin='lower', 
            aspect='auto',
            extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
            cmap='viridis'
        )
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label('Density')
        
        # Add contour lines
        x_centers = (x_edges[:-1] + x_edges[1:]) / 2
        y_centers = (y_edges[:-1] + y_edges[1:]) / 2
        X, Y = np.meshgrid(x_centers, y_centers)
        
        # Smooth density for contour lines
        from scipy.ndimage import gaussian_filter
        h_smooth = gaussian_filter(h, sigma=1.0)
        
        # Add contour lines
        contour = ax.contour(X, Y, h_smooth, colors='white', alpha=0.5, linewidths=0.5)
        
        # Calculate correlation
        corr, p = pearsonr(df[x_var], df[y_var])
        
        # Add correlation text
        ax.annotate(
            f"r = {corr:.3f}, p = {p:.3e}",
            xy=(0.05, 0.95),
            xycoords='axes fraction',
            fontsize=12,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="grey", alpha=0.8)
        )
        
        # Add a line representing the linear regression
        slope, intercept, _, _, _ = stats.linregress(df[x_var], df[y_var])
        x_range = np.array([df[x_var].min(), df[x_var].max()])
        y_range = intercept + slope * x_range
        ax.plot(x_range, y_range, 'r-', lw=2)
        
        ax.set_xlabel(x_var)
        ax.set_ylabel(y_var)
        ax.set_title(f'2D Density Heatmap: {y_var} vs {x_var}')
        
        plt.tight_layout()
        if filename:
            plt.savefig(os.path.join(output_dir, filename), dpi=dpi)
            plt.close()
        
        return {
            'correlation': corr,
            'p_value': p,
            'slope': slope,
            'intercept': intercept
        }
    
    # Create density heatmaps
    heatmap_corrn_gOSI = create_2d_density_heatmap(
        clean_df, 'corrn', 'gOSI', n_bins=15,
        filename='heatmap_corrn_gOSI.png'
    )
    
    heatmap_corrn_volume = create_2d_density_heatmap(
        clean_df, 'corrn', 'volume', n_bins=15,
        filename='heatmap_corrn_volume.png'
    )
    
    # 3.6 Box Plots by Quartile Analysis
    # ---------------------------------
    log("Creating quartile-based box plots...")
    
    def create_quartile_boxplots(df, x_var, y_var, filename=None):
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Create quartile groups
        df = df.copy()
        df[f'{x_var}_quartile'] = pd.qcut(df[x_var], 4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])
        
        # Create boxplot
        sns.boxplot(
            x=f'{x_var}_quartile', 
            y=y_var, 
            data=df, 
            ax=ax,
            palette='viridis'
        )
        
        # Add individual data points
        sns.stripplot(
            x=f'{x_var}_quartile', 
            y=y_var, 
            data=df,
            ax=ax,
            size=4,
            color='black',
            alpha=0.3,
            jitter=True
        )
        
        # Add mean markers
        means = df.groupby(f'{x_var}_quartile')[y_var].mean()
        ax.plot(range(len(means)), means.values, 'ro-', linewidth=2, markersize=8)
        
        # Calculate statistics
        quartile_stats = df.groupby(f'{x_var}_quartile')[y_var].agg(['mean', 'median', 'std', 'count'])
        
        # Perform ANOVA
        groups = [df[df[f'{x_var}_quartile'] == q][y_var].values for q in ['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']]
        f_val, p_val = f_oneway(*groups)
        
        # Add ANOVA results
        ax.set_title(f'{y_var} by {x_var} Quartiles (ANOVA: F={f_val:.2f}, p={p_val:.3e})')
        
        # Add count labels
        for i, (_, stats) in enumerate(quartile_stats.iterrows()):
            ax.annotate(
                f'n={int(stats["count"])}',
                xy=(i, ax.get_ylim()[1] * 0.9),
                ha='center',
                fontsize=10
            )
        
        ax.set_xlabel(f'{x_var} Quartiles')
        ax.set_ylabel(y_var)
        
        plt.tight_layout()
        if filename:
            plt.savefig(os.path.join(output_dir, filename), dpi=dpi)
            plt.close()
        
        return {
            'quartile_stats': quartile_stats,
            'anova_f': f_val,
            'anova_p': p_val
        }
    
    # Create quartile boxplots
    quartile_gOSI_corrn = create_quartile_boxplots(
        clean_df, 'gOSI', 'corrn',
        filename='quartile_gOSI_corrn.png'
    )
    
    quartile_corrn_gOSI = create_quartile_boxplots(
        clean_df, 'corrn', 'gOSI',
        filename='quartile_corrn_gOSI.png'
    )
    
    quartile_volume_corrn = create_quartile_boxplots(
        clean_df, 'volume', 'corrn',
        filename='quartile_volume_corrn.png'
    )
    
    # 4. Advanced Analysis
    # ===================
    log("Performing advanced analysis...")
    
    # 4.1 Sliding Window Correlation Analysis
    # --------------------------------------
    log("Performing sliding window correlation analysis...")
    
    def sliding_window_correlation(df, x_var, y_var, window_size=20, step=5, filename=None):
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Sort data by x variable
        sorted_df = df.sort_values(by=x_var).reset_index(drop=True)
        
        # Initialize lists for statistics
        window_centers = []
        correlations = []
        p_values = []
        window_sizes = []
        
        # Calculate correlations in sliding windows
        for i in range(0, len(sorted_df) - window_size, step):
            window = sorted_df.iloc[i:i+window_size]
            
            # Calculate window center
            window_center = window[x_var].mean()
            
            # Calculate Pearson correlation
            corr, p = pearsonr(window[x_var], window[y_var])
            
            window_centers.append(window_center)
            correlations.append(corr)
            p_values.append(p)
            window_sizes.append(len(window))
        
        # Create a DataFrame for results
        results = pd.DataFrame({
            'window_center': window_centers,
            'correlation': correlations,
            'p_value': p_values,
            'window_size': window_sizes
        })
        
        # Plot correlation by window center
        ax.plot(results['window_center'], results['correlation'], 'o-', 
               markersize=6, linewidth=2, color=colors[0])
        
        # Add horizontal line at zero correlation
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
        
        # Highlight significant correlations
        significant = results['p_value'] < 0.05
        if any(significant):
            ax.scatter(
                results.loc[significant, 'window_center'],
                results.loc[significant, 'correlation'],
                color='red',
                s=80,
                marker='*',
                label='p < 0.05'
            )
            ax.legend()
        
        # Shade area between correlation curve and zero
        ax.fill_between(
            results['window_center'],
            0,
            results['correlation'],
            alpha=0.2,
            color=colors[0]
        )
        
        # Add labels and title
        ax.set_xlabel(f'{x_var} (window center)')
        ax.set_ylabel(f'Correlation with {y_var}')
        ax.set_title(f'Sliding Window Correlation Analysis (window size = {window_size})')
        
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        if filename:
            plt.savefig(os.path.join(output_dir, filename), dpi=dpi)
            plt.close()
        
        return results

    # Perform sliding window analysis
    sliding_corrn_gOSI = sliding_window_correlation(
        clean_df, 'corrn', 'gOSI', window_size=25, step=5,
        filename='sliding_window_corrn_gOSI.png'
    )
    
    sliding_gOSI_corrn = sliding_window_correlation(
        clean_df, 'gOSI', 'corrn', window_size=25, step=5,
        filename='sliding_window_gOSI_corrn.png'
    )
    
    # 4.2 3D Visualization Analysis
    # ----------------------------
    log("Creating 3D visualizations...")
    
    def create_3d_scatter(df, x_var, y_var, z_var, color_var=None, filename=None):
        fig = plt.figure(figsize=(12, 10))
        ax = fig.add_subplot(111, projection='3d')
        
        # Create scatter plot
        if color_var:
            scatter = ax.scatter(
                df[x_var],
                df[y_var],
                df[z_var],
                c=df[color_var],
                cmap='viridis',
                s=40,
                alpha=0.7
            )
            cbar = plt.colorbar(scatter, ax=ax, pad=0.1)
            cbar.set_label(color_var)
        else:
            ax.scatter(
                df[x_var],
                df[y_var],
                df[z_var],
                s=40,
                alpha=0.7,
                color=colors[0]
            )
        
        # Add a best-fit plane if needed
        if x_var == 'corrn' and y_var == 'gOSI' and z_var == 'volume':
            # Create a meshgrid for the plane
            x_range = np.linspace(df[x_var].min(), df[x_var].max(), 10)
            y_range = np.linspace(df[y_var].min(), df[y_var].max(), 10)
            X, Y = np.meshgrid(x_range, y_range)
            
            # Fit a plane (z = ax + by + c)
            A = np.column_stack((df[x_var], df[y_var], np.ones(len(df))))
            B = df[z_var]
            coeffs, residuals, _, _ = np.linalg.lstsq(A, B, rcond=None)
            
            # Calculate z values for the plane
            Z = coeffs[0] * X + coeffs[1] * Y + coeffs[2]
            
            # Plot the plane
            ax.plot_surface(X, Y, Z, alpha=0.3, color='gray')
            
            # Add equation
            eqn = f"volume = {coeffs[0]:.2f} × corrn + {coeffs[1]:.2f} × gOSI + {coeffs[2]:.2f}"
            ax.text2D(0.05, 0.95, eqn, transform=ax.transAxes, fontsize=12,
                     bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
        
        # Set labels and title
        ax.set_xlabel(x_var)
        ax.set_ylabel(y_var)
        ax.set_zlabel(z_var)
        ax.set_title(f'3D Relationship between {x_var}, {y_var}, and {z_var}')
        
        # Add grid lines
        ax.grid(True, alpha=0.3)
        
        # Improve perspective
        ax.view_init(elev=30, azim=45)
        
        plt.tight_layout()
        if filename:
            plt.savefig(os.path.join(output_dir, filename), dpi=dpi)
            plt.close()
        
        return fig
    
    # Create 3D visualizations
    create_3d_scatter(
        clean_df, 'corrn', 'gOSI', 'volume', 
        filename='3d_corrn_gOSI_volume.png'
    )
    
    create_3d_scatter(
        clean_df, 'corrn', 'gOSI', 'volume', color_var='pref_ori_norm',
        filename='3d_corrn_gOSI_volume_by_orientation.png'
    )
    
    # 4.3 Machine Learning-Based Feature Importance
    # -------------------------------------------
    log("Analyzing feature importance with machine learning...")
    
    def analyze_feature_importance(df, target='corrn', n_estimators=100, filename=None):
        # Select relevant features
        features = ['volume', 'pref_ori_norm', 'gOSI', 'gDSI']
        if 'cc_abs' in df.columns:
            features.append('cc_abs')
        
        # Drop rows with NaN values
        valid_data = df.dropna(subset=features + [target])
        
        X = valid_data[features]
        y = valid_data[target]
        
        # Split data for validation
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)
        
        # Create and train random forest model
        rf = RandomForestRegressor(n_estimators=n_estimators, random_state=seed)
        rf.fit(X_train, y_train)
        
        # Make predictions
        y_pred = rf.predict(X_test)
        
        # Calculate evaluation metrics
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        
        # Get feature importances
        importances = rf.feature_importances_
        
        # Create DataFrame for easier plotting
        importance_df = pd.DataFrame({
            'Feature': features,
            'Importance': importances
        }).sort_values('Importance', ascending=False)
        
        # Create plot
        fig, ax = plt.subplots(figsize=(10, 6))
        
        sns.barplot(
            x='Importance',
            y='Feature',
            data=importance_df,
            palette='viridis',
            ax=ax
        )
        
        # Add labels and title
        ax.set_xlabel('Relative Importance')
        ax.set_ylabel('Feature')
        ax.set_title(f'Feature Importance for Predicting {target} (R² = {r2:.3f}, RMSE = {rmse:.3f})')
        
        # Add grid
        ax.grid(True, axis='x', alpha=0.3)
        
        plt.tight_layout()
        if filename:
            plt.savefig(os.path.join(output_dir, filename), dpi=dpi)
            plt.close()
        
        # Create model performance plot (predicted vs actual)
        fig, ax = plt.subplots(figsize=(8, 8))
        
        ax.scatter(y_test, y_pred, alpha=0.6, s=30)
        
        # Add diagonal line
        diag_line = np.linspace(min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max()), 100)
        ax.plot(diag_line, diag_line, 'r--', linewidth=2)
        
        # Add labels and title
        ax.set_xlabel(f'Actual {target}')
        ax.set_ylabel(f'Predicted {target}')
        ax.set_title(f'Random Forest Model Performance\nR² = {r2:.3f}, RMSE = {rmse:.3f}')
        
        # Add grid
        ax.grid(True, alpha=0.3)
        
        # Make axes equal
        ax.set_aspect('equal')
        
        plt.tight_layout()
        if filename:
            model_filename = filename.replace('.png', '_performance.png')
            plt.savefig(os.path.join(output_dir, model_filename), dpi=dpi)
            plt.close()
        
        return {
            'feature_importance': importance_df,
            'r2': r2,
            'rmse': rmse,
            'model': rf
        }
    
    # Analyze feature importance
    feature_importance_results = analyze_feature_importance(
        clean_df, target='corrn', n_estimators=100,
        filename='feature_importance.png'
    )
    
    # 4.4 Cluster Analysis of Neurons
    # -----------------------------
    log("Performing cluster analysis of neurons...")
    
    def perform_cluster_analysis(df, features, n_clusters=3, filename_prefix='clustering'):
        # Select and prepare data
        valid_data = df.dropna(subset=features).copy()
        X = valid_data[features]
        
        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Determine optimal number of clusters using silhouette score
        from sklearn.metrics import silhouette_score
        
        silhouette_scores = []
        k_range = range(2, min(8, len(X) // 10))  # Limit max clusters
        
        for k in k_range:
            kmeans = KMeans(n_clusters=k, random_state=seed, n_init=10)
            labels = kmeans.fit_predict(X_scaled)
            silhouette_scores.append(silhouette_score(X_scaled, labels))
        
        # Plot silhouette scores
        fig, ax = plt.subplots(figsize=(10, 6))
        
        ax.plot(list(k_range), silhouette_scores, 'o-', linewidth=2, markersize=8)
        ax.set_xlabel('Number of Clusters (k)')
        ax.set_ylabel('Silhouette Score')
        ax.set_title('Optimal Number of Clusters Determination')
        ax.grid(True, alpha=0.3)
        
        # Find optimal k
        optimal_k = k_range[np.argmax(silhouette_scores)]
        ax.axvline(optimal_k, color='red', linestyle='--')
        ax.text(optimal_k + 0.1, min(silhouette_scores), f'Optimal k = {optimal_k}', 
                color='red', va='bottom')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{filename_prefix}_silhouette.png'), dpi=dpi)
        plt.close()
        
        # Use optimal k for clustering (or user-provided n_clusters if specified)
        k = optimal_k if n_clusters is None else n_clusters
        
        # Perform K-means clustering
        kmeans = KMeans(n_clusters=k, random_state=seed, n_init=10)
        valid_data['cluster'] = kmeans.fit_predict(X_scaled)
        
        # Get cluster centers (in original feature space)
        centers = scaler.inverse_transform(kmeans.cluster_centers_)
        
        # Create a pairplot to visualize clusters
        if len(features) >= 2:
            # Select a subset of features if there are many
            plot_features = features[:4] if len(features) > 4 else features
            
            # Add 'cluster' for coloring
            plot_data = valid_data[plot_features + ['cluster']].copy()
            
            # Convert cluster to categorical for better coloring
            plot_data['cluster'] = plot_data['cluster'].astype('category')
            
            # Create pairplot
            g = sns.pairplot(
                plot_data,
                hue='cluster',
                palette='viridis',
                diag_kind='kde',
                plot_kws={'alpha': 0.6, 's': 30},
                diag_kws={'alpha': 0.6},
                corner=True
            )
            
            g.fig.suptitle(f'Cluster Analysis of Neurons (k={k})', y=1.02, fontsize=16)
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f'{filename_prefix}_pairplot.png'), dpi=dpi)
            plt.close()
        
        # If we have enough features, create a t-SNE visualization
        if len(X_scaled) >= 50:  # Need enough samples for t-SNE
            from sklearn.manifold import TSNE
            
            # Create t-SNE embedding
            tsne = TSNE(n_components=2, random_state=seed, perplexity=min(30, len(X_scaled) // 5))
            tsne_results = tsne.fit_transform(X_scaled)
            
            # Create t-SNE plot
            fig, ax = plt.subplots(figsize=(10, 8))
            
            scatter = ax.scatter(
                tsne_results[:, 0],
                tsne_results[:, 1],
                c=valid_data['cluster'],
                cmap='viridis',
                alpha=0.7,
                s=50
            )
            
            # Add legend
            legend1 = ax.legend(*scatter.legend_elements(),
                              title="Clusters")
            ax.add_artist(legend1)
            
            ax.set_xlabel('t-SNE Dimension 1')
            ax.set_ylabel('t-SNE Dimension 2')
            ax.set_title(f't-SNE Visualization of Neuron Clusters (k={k})')
            
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f'{filename_prefix}_tsne.png'), dpi=dpi)
            plt.close()
        
        # Calculate cluster statistics
        cluster_stats = valid_data.groupby('cluster').agg({
            features[0]: ['count', 'mean', 'std'],
            **{feature: ['mean', 'std'] for feature in features[1:]}
        })
        
        # Test for significant differences between clusters
        anova_results = {}
        for feature in features:
            groups = [valid_data[valid_data['cluster'] == i][feature].values for i in range(k)]
            f_val, p_val = f_oneway(*groups)
            anova_results[feature] = {'F': f_val, 'p': p_val}
        
        return {
            'clustered_data': valid_data,
            'optimal_k': optimal_k,
            'cluster_centers': centers,
            'cluster_stats': cluster_stats,
            'anova_results': anova_results
        }
    
    # Perform cluster analysis
    cluster_results = perform_cluster_analysis(
        clean_df,
        features=['corrn', 'gOSI', 'volume', 'pref_ori_norm'],
        filename_prefix='neuron_clusters'
    )
    
    # 4.5 Joint Distribution Analysis
    # -----------------------------
    log("Analyzing joint distributions...")
    
    def create_joint_distribution_plot(df, x_var, y_var, filename=None):
        # Create joint plot
        g = sns.jointplot(
            x=x_var,
            y=y_var,
            data=df,
            kind='scatter',
            height=8,
            ratio=3,
            space=0.2,
            color=colors[0],
            alpha=0.6,
            s=30
        )
        
        # Add regression line with confidence interval
        g.plot_joint(sns.regplot, scatter=False, line_kws={'color': 'red'})
        
        # Calculate correlation
        corr, p = pearsonr(df[x_var], df[y_var])
        
        # Add correlation annotation
        g.fig.text(
            0.65, 0.15,
            f"r = {corr:.3f}\np = {p:.3e}",
            fontsize=12,
            bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.5')
        )
        
        # Add contour plot on top of points
        cmap = sns.cubehelix_palette(start=.5, rot=-.5, as_cmap=True)
        g.plot_joint(sns.kdeplot, levels=5, color='blue', linewidths=0.5, alpha=0.5)
        
        # Add marginal histograms with KDE
        g.plot_marginals(sns.histplot, kde=True, color=colors[0], alpha=0.6)
        
        # Set axis labels
        g.set_axis_labels(x_var, y_var, fontsize=12)
        
        # Set title
        g.fig.suptitle(f'Joint Distribution of {y_var} vs {x_var}', y=1.02, fontsize=14)
        
        if filename:
            plt.savefig(os.path.join(output_dir, filename), dpi=dpi)
            plt.close()
        
        return g
    
    # Create joint distribution plots
    create_joint_distribution_plot(
        clean_df, 'corrn', 'gOSI',
        filename='joint_distribution_corrn_gOSI.png'
    )
    
    create_joint_distribution_plot(
        clean_df, 'corrn', 'volume',
        filename='joint_distribution_corrn_volume.png'
    )
    
    # 4.6 Bootstrap Confidence Intervals
    # --------------------------------
    log("Calculating bootstrap confidence intervals...")
    
    def bootstrap_confidence_intervals(df, x_var, y_var, n_bootstrap=1000, filename=None):
        # Drop NaN values
        valid_data = df.dropna(subset=[x_var, y_var])
        
        # Calculate observed correlation
        observed_corr, observed_p = pearsonr(valid_data[x_var], valid_data[y_var])
        
        # Initialize empty array for bootstrap correlations
        bootstrap_corrs = np.zeros(n_bootstrap)
        
        # Get data as arrays
        x = valid_data[x_var].values
        y = valid_data[y_var].values
        n = len(x)
        
        # Perform bootstrap resampling
        for i in range(n_bootstrap):
            # Resample with replacement
            indices = np.random.randint(0, n, size=n)
            x_resampled = x[indices]
            y_resampled = y[indices]
            
            # Calculate correlation for this resample
            bootstrap_corrs[i], _ = pearsonr(x_resampled, y_resampled)
        
        # Calculate confidence intervals
        ci_lower = np.percentile(bootstrap_corrs, 2.5)
        ci_upper = np.percentile(bootstrap_corrs, 97.5)
        
        # Create visualization
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Plot bootstrap distribution
        sns.histplot(bootstrap_corrs, kde=True, bins=30, ax=ax, color=colors[0], alpha=0.6)
        
        # Add vertical lines
        ax.axvline(observed_corr, color='red', linestyle='-', linewidth=2,
                  label=f'Observed r = {observed_corr:.3f}')
        ax.axvline(ci_lower, color='blue', linestyle='--', linewidth=2,
                  label=f'95% CI Lower = {ci_lower:.3f}')
        ax.axvline(ci_upper, color='blue', linestyle='--', linewidth=2,
                  label=f'95% CI Upper = {ci_upper:.3f}')
        
        # Add labels and title
        ax.set_xlabel('Correlation Coefficient (r)')
        ax.set_ylabel('Frequency')
        ax.set_title(f'Bootstrap Distribution of Correlation between {x_var} and {y_var}\n({n_bootstrap} resamples)')
        
        # Add legend
        ax.legend()
        
        plt.tight_layout()
        if filename:
            plt.savefig(os.path.join(output_dir, filename), dpi=dpi)
            plt.close()
        
        return {
            'observed_corr': observed_corr,
            'observed_p': observed_p,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'bootstrap_samples': bootstrap_corrs
        }
    
    # Perform bootstrap analysis
    bootstrap_corrn_gOSI = bootstrap_confidence_intervals(
        clean_df, 'corrn', 'gOSI', n_bootstrap=1000,
        filename='bootstrap_corrn_gOSI.png'
    )
    
    bootstrap_corrn_volume = bootstrap_confidence_intervals(
        clean_df, 'corrn', 'volume', n_bootstrap=1000,
        filename='bootstrap_corrn_volume.png'
    )
    
    # 5. Integrated Analysis Dashboard
    # ===============================
    log("Creating integrated analysis dashboard...")
    
    def create_dashboard(df, results_dict, filename='dashboard.png'):
        # Create a large figure with subplots for key visualizations
        fig = plt.figure(figsize=(24, 18))
        gs = gridspec.GridSpec(3, 4, figure=fig)
        
        # 1. Correlation matrix heatmap (top-left)
        ax1 = fig.add_subplot(gs[0, 0])
        corr_matrix = df[['corrn', 'volume', 'pref_ori_norm', 'gOSI', 'gDSI']].corr()
        
        sns.heatmap(
            corr_matrix,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            ax=ax1
        )
        ax1.set_title('Correlation Matrix')
        
        # 2. RNN Correlation vs gOSI (top, second column)
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.scatter(df['corrn'], df['gOSI'], alpha=0.6, s=30, color=colors[0])
        
        # Add regression line
        slope, intercept, r_value, p_value, _ = stats.linregress(df['corrn'], df['gOSI'])
        x_range = np.linspace(df['corrn'].min(), df['corrn'].max(), 100)
        ax2.plot(x_range, intercept + slope * x_range, 'r-', linewidth=2)
        
        # Add correlation text
        ax2.text(
            0.05, 0.95,
            f"r = {r_value:.3f}, p = {p_value:.3e}",
            transform=ax2.transAxes,
            fontsize=10,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
        )
        
        ax2.set_xlabel('RNN Correlation')
        ax2.set_ylabel('gOSI')
        ax2.set_title('Orientation Selectivity vs RNN Correlation')
        
        # 3. Binned analysis of gOSI by correlation (top, third column)
        ax3 = fig.add_subplot(gs[0, 2])
        
        # Create bins
        n_bins = 8
        bins = np.linspace(df['corrn'].min(), df['corrn'].max(), n_bins + 1)
        bin_centers = [(bins[i] + bins[i+1]) / 2 for i in range(len(bins) - 1)]
        
        # Initialize arrays for means and errors
        means = []
        errors = []
        
        # Calculate statistics for each bin
        for i in range(len(bins) - 1):
            mask = (df['corrn'] >= bins[i]) & (df['corrn'] < bins[i+1])
            bin_data = df[mask]['gOSI']
            
            means.append(bin_data.mean() if len(bin_data) > 0 else np.nan)
            errors.append(bin_data.std() / np.sqrt(len(bin_data)) if len(bin_data) > 1 else np.nan)
        
        # Plot binned means with error bars
        ax3.errorbar(bin_centers, means, yerr=errors, fmt='o-', linewidth=2, markersize=8, 
                   capsize=5, color=colors[0])
        
        ax3.set_xlabel('RNN Correlation')
        ax3.set_ylabel('Mean gOSI')
        ax3.set_title('Binned Analysis of gOSI by Correlation')
        
        # 4. Session-specific correlations (top-right)
        ax4 = fig.add_subplot(gs[0, 3])
        
        # Get session data
        sessions = []
        corrs = []
        
        for session in sorted(df['session'].unique()):
            session_data = df[df['session'] == session]
            corr, _ = pearsonr(session_data['corrn'], session_data['gOSI'])
            
            sessions.append(f"S{session}")
            corrs.append(corr)
        
        # Plot bar chart
        bars = ax4.bar(sessions, corrs, color=[session_colors[i % len(session_colors)] for i in range(len(sessions))])
        
        # Add horizontal line at zero
        ax4.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        
        ax4.set_xlabel('Session')
        ax4.set_ylabel('Correlation (r)')
        ax4.set_title('RNN-gOSI Correlation by Session')
        
        # 5. Feature importance (middle-left)
        ax5 = fig.add_subplot(gs[1, 0])
        
        # Get feature importance data
        importance_df = results_dict.get('feature_importance_results', {}).get('feature_importance')
        
        if importance_df is not None:
            sns.barplot(
                x='Importance',
                y='Feature',
                data=importance_df,
                palette='viridis',
                ax=ax5
            )
            
            ax5.set_xlabel('Relative Importance')
            ax5.set_ylabel('Feature')
            ax5.set_title('Feature Importance for Predicting RNN Correlation')
        else:
            ax5.text(0.5, 0.5, "Feature importance data not available", 
                    ha='center', va='center', fontsize=12)
        
        # 6. Orientation polar plot (middle, second column)
        ax6 = fig.add_subplot(gs[1, 1], projection='polar')
        
        # Create orientation bins (in radians)
        orientation_bins = np.linspace(0, np.pi, 9)  # 8 bins spanning 0 to π
        bin_centers = [(orientation_bins[i] + orientation_bins[i+1]) / 2 for i in range(len(orientation_bins) - 1)]
        
        # Calculate mean correlation for each orientation bin
        ori_means = []
        
        for i in range(len(orientation_bins) - 1):
            mask = (df['pref_ori_norm'] >= orientation_bins[i]) & (df['pref_ori_norm'] < orientation_bins[i+1])
            bin_data = df[mask]['corrn']
            
            ori_means.append(bin_data.mean() if len(bin_data) > 0 else 0)
        
        # Plot on polar axis
        bars = ax6.bar(
            bin_centers, 
            ori_means, 
            width=np.pi/8, 
            alpha=0.7,
            color=plt.cm.viridis(np.linspace(0, 1, len(bin_centers)))
        )
        
        # Set axis limits and labels
        ax6.set_theta_zero_location('N')
        ax6.set_theta_direction(-1)
        ax6.set_rlabel_position(0)
        
        # Set custom theta labels
        ax6.set_xticks(np.linspace(0, np.pi, 5))
        ax6.set_xticklabels(['0°', '45°', '90°', '135°', '180°'])
        
        ax6.set_title('RNN Correlation by Orientation', pad=20)
        
        # 7. gOSI quartiles boxplot (middle, third column)
        ax7 = fig.add_subplot(gs[1, 2])
        
        # Create quartile groups
        df_copy = df.copy()
        df_copy['gOSI_quartile'] = pd.qcut(df_copy['gOSI'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
        
        # Create boxplot
        sns.boxplot(
            x='gOSI_quartile',
            y='corrn',
            data=df_copy,
            palette='viridis',
            ax=ax7
        )
        
        # Add individual data points
        sns.stripplot(
            x='gOSI_quartile',
            y='corrn',
            data=df_copy,
            color='black',
            size=3,
            alpha=0.3,
            jitter=True,
            ax=ax7
        )
        
        ax7.set_xlabel('gOSI Quartile')
        ax7.set_ylabel('RNN Correlation')
        ax7.set_title('RNN Correlation by gOSI Quartile')
        
        # 8. 2D density heatmap (middle-right)
        ax8 = fig.add_subplot(gs[1, 3])
        
        # Create 2D histogram
        h, x_edges, y_edges = np.histogram2d(
            df['corrn'],
            df['gOSI'],
            bins=15,
            density=True
        )
        
        # Convert to density
        h = h.T  # Transpose for correct orientation
        
        # Plot heatmap
        im = ax8.imshow(
            h,
            origin='lower',
            aspect='auto',
            extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
            cmap='viridis'
        )
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax8)
        cbar.set_label('Density')
        
        # Add contour lines
        x_centers = (x_edges[:-1] + x_edges[1:]) / 2
        y_centers = (y_edges[:-1] + y_edges[1:]) / 2
        X, Y = np.meshgrid(x_centers, y_centers)
        
        # Add contour lines
        ax8.contour(X, Y, h, colors='white', alpha=0.5, linewidths=0.5)
        
        ax8.set_xlabel('RNN Correlation')
        ax8.set_ylabel('gOSI')
        ax8.set_title('2D Density Map')
        
        # 9. 3D scatter plot (bottom-left)
        ax9 = fig.add_subplot(gs[2, 0], projection='3d')
        
        scatter = ax9.scatter(
            df['corrn'],
            df['gOSI'],
            df['volume'],
            c=df['pref_ori_norm'],
            cmap='hsv',
            s=30,
            alpha=0.7
        )
        
        cbar = plt.colorbar(scatter, ax=ax9)
        cbar.set_label('Orientation (rad)')
        
        ax9.set_xlabel('RNN Correlation')
        ax9.set_ylabel('gOSI')
        ax9.set_zlabel('Volume')
        ax9.set_title('3D Relationship')
        
        # 10. Clustered data visualization (bottom, second column)
        ax10 = fig.add_subplot(gs[2, 1])
        
        # Get clustered data if available
        clustered_data = results_dict.get('cluster_results', {}).get('clustered_data')
        
        if clustered_data is not None:
            scatter = ax10.scatter(
                clustered_data['corrn'],
                clustered_data['gOSI'],
                c=clustered_data['cluster'],
                cmap='viridis',
                s=40,
                alpha=0.7
            )
            
            # Create legend
            legend = ax10.legend(*scatter.legend_elements(),
                               title="Clusters")
            ax10.add_artist(legend)
            
            ax10.set_xlabel('RNN Correlation')
            ax10.set_ylabel('gOSI')
            ax10.set_title('Neuron Clusters')
        else:
            ax10.text(0.5, 0.5, "Cluster data not available", 
                    ha='center', va='center', fontsize=12)
        
        # 11. Bootstrap distribution (bottom, third column)
        ax11 = fig.add_subplot(gs[2, 2])
        
        # Get bootstrap data if available
        bootstrap_samples = results_dict.get('bootstrap_corrn_gOSI', {}).get('bootstrap_samples')
        
        if bootstrap_samples is not None:
            observed_corr = results_dict['bootstrap_corrn_gOSI']['observed_corr']
            ci_lower = results_dict['bootstrap_corrn_gOSI']['ci_lower']
            ci_upper = results_dict['bootstrap_corrn_gOSI']['ci_upper']
            
            # Plot bootstrap distribution
            sns.histplot(bootstrap_samples, kde=True, bins=30, ax=ax11, color=colors[0], alpha=0.6)
            
            # Add vertical lines
            ax11.axvline(observed_corr, color='red', linestyle='-', linewidth=2,
                      label=f'Observed r = {observed_corr:.3f}')
            ax11.axvline(ci_lower, color='blue', linestyle='--', linewidth=2,
                      label=f'95% CI Lower = {ci_lower:.3f}')
            ax11.axvline(ci_upper, color='blue', linestyle='--', linewidth=2,
                      label=f'95% CI Upper = {ci_upper:.3f}')
            
            ax11.set_xlabel('Correlation Coefficient (r)')
            ax11.set_ylabel('Frequency')
            ax11.set_title('Bootstrap Distribution\n(corrn vs gOSI)')
            ax11.legend(fontsize=8)
        else:
            ax11.text(0.5, 0.5, "Bootstrap data not available", 
                    ha='center', va='center', fontsize=12)
        
        # 12. Sliding window correlation (bottom-right)
        ax12 = fig.add_subplot(gs[2, 3])
        
        # Get sliding window data if available
        sliding_results = results_dict.get('sliding_corrn_gOSI')
        
        if sliding_results is not None:
            # Plot correlation by window center
            ax12.plot(sliding_results['window_center'], sliding_results['correlation'], 
                   'o-', markersize=6, linewidth=2, color=colors[0])
            
            # Add horizontal line at zero correlation
            ax12.axhline(y=0, color='gray', linestyle='--', alpha=0.7)
            
            # Highlight significant correlations
            significant = sliding_results['p_value'] < 0.05
            if any(significant):
                ax12.scatter(
                    sliding_results.loc[significant, 'window_center'],
                    sliding_results.loc[significant, 'correlation'],
                    color='red',
                    s=80,
                    marker='*',
                    label='p < 0.05'
                )
                ax12.legend(fontsize=8)
            
            # Shade area between correlation curve and zero
            ax12.fill_between(
                sliding_results['window_center'],
                0,
                sliding_results['correlation'],
                alpha=0.2,
                color=colors[0]
            )
            
            ax12.set_xlabel('Correlation (window center)')
            ax12.set_ylabel('Correlation with gOSI')
            ax12.set_title('Sliding Window Correlation')
        else:
            ax12.text(0.5, 0.5, "Sliding window data not available", 
                    ha='center', va='center', fontsize=12)
        
        # Add overall title
        fig.suptitle('Neural Correlation Analysis Dashboard', fontsize=24, y=0.98)
        
        # Adjust layout
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        
        # Save figure
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
        return fig
    
    # Create integrated dashboard
    dashboard = create_dashboard(
        clean_df,
        {
            'feature_importance_results': feature_importance_results,
            'cluster_results': cluster_results,
            'bootstrap_corrn_gOSI': bootstrap_corrn_gOSI,
            'sliding_corrn_gOSI': sliding_corrn_gOSI
        },
        filename='neural_analysis_dashboard.png'
    )
    
    # 6. Summary Statistics and Interpretations
    # =======================================
    log("Generating summary report...")
    
    # Create a summary report with key findings
    def generate_summary_report():
        # Create summary text
        summary = [
            "# Neural Correlation Analysis - Summary Report",
            f"\nDate: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"\nDataset Overview:",
            f"- Total neurons analyzed: {len(clean_df)}",
            f"- Number of sessions: {clean_df['session'].nunique()}",
            f"- Number of scans: {clean_df['scan'].nunique()}"
        ]
        
        # Add correlation statistics
        summary.append("\n## Correlation Statistics")
        
        summary.append("\n### Overall Correlations:")
        for var in ['gOSI', 'volume', 'pref_ori_norm']:
            corr, p = pearsonr(clean_df['corrn'], clean_df[var])
            sig_text = "Significant" if p < 0.05 else "Not significant"
            summary.append(f"- RNN Correlation with {var}: r = {corr:.3f}, p = {p:.3e} ({sig_text})")
        
        # Add session-specific findings
        summary.append("\n### Session-Specific Findings:")
        for session in sorted(clean_df['session'].unique()):
            session_data = clean_df[clean_df['session'] == session]
            corr, p = pearsonr(session_data['corrn'], session_data['gOSI'])
            sig_text = "Significant" if p < 0.05 else "Not significant"
            summary.append(f"- Session {session} (n={len(session_data)}): RNN-gOSI correlation = {corr:.3f}, p = {p:.3e} ({sig_text})")
        
        # Add regression results
        summary.append("\n## Linear Regression Results")
        
        if 'reg_corrn_gOSI' in locals():
            summary.append("\n### RNN Correlation vs gOSI:")
            summary.append(f"- Slope: {reg_corrn_gOSI['slope']:.3f}")
            summary.append(f"- Intercept: {reg_corrn_gOSI['intercept']:.3f}")
            summary.append(f"- R²: {reg_corrn_gOSI['r_squared']:.3f}")
            summary.append(f"- p-value: {reg_corrn_gOSI['p_value']:.3e}")
        
        if 'reg_corrn_volume' in locals():
            summary.append("\n### RNN Correlation vs Volume:")
            summary.append(f"- Slope: {reg_corrn_volume['slope']:.3f}")
            summary.append(f"- Intercept: {reg_corrn_volume['intercept']:.3f}")
            summary.append(f"- R²: {reg_corrn_volume['r_squared']:.3f}")
            summary.append(f"- p-value: {reg_corrn_volume['p_value']:.3e}")
        

        if feature_importance_results:
            summary.append("\n## Feature Importance for Predicting RNN Correlation")
            summary.append(f"- Model performance: R² = {feature_importance_results['r2']:.3f}, RMSE = {feature_importance_results['rmse']:.3f}")
            summary.append("\n### Relative Feature Importance:")
            
            for _, row in feature_importance_results['feature_importance'].iterrows():
                summary.append(f"- {row['Feature']}: {row['Importance']:.3f}")
        
        if cluster_results:
            summary.append("\n## Cluster Analysis Results")
            summary.append(f"- Optimal number of clusters: {cluster_results['optimal_k']}")
            summary.append(f"- Number of neurons in each cluster:")
            
            cluster_counts = cluster_results['clustered_data']['cluster'].value_counts().sort_index()
            for cluster, count in cluster_counts.items():
                summary.append(f"  - Cluster {cluster}: {count} neurons")
            
            summary.append("\n### ANOVA Results for Differences Between Clusters:")
            for feature, stats in cluster_results['anova_results'].items():
                sig_text = "Significant" if stats['p'] < 0.05 else "Not significant"
                summary.append(f"- {feature}: F = {stats['F']:.3f}, p = {stats['p']:.3e} ({sig_text})")
        
        if bootstrap_corrn_gOSI:
            summary.append("\n## Bootstrap Confidence Intervals")
            
            summary.append("\n### RNN Correlation vs gOSI:")
            summary.append(f"- Observed correlation: {bootstrap_corrn_gOSI['observed_corr']:.3f}")
            summary.append(f"- 95% CI: [{bootstrap_corrn_gOSI['ci_lower']:.3f}, {bootstrap_corrn_gOSI['ci_upper']:.3f}]")
            
            if bootstrap_corrn_volume:
                summary.append("\n### RNN Correlation vs Volume:")
                summary.append(f"- Observed correlation: {bootstrap_corrn_volume['observed_corr']:.3f}")
                summary.append(f"- 95% CI: [{bootstrap_corrn_volume['ci_lower']:.3f}, {bootstrap_corrn_volume['ci_upper']:.3f}]")
        
        summary.append("\n## Key Interpretations")
        
        # Overall relationship between RNN correlation and gOSI
        corr, p = pearsonr(clean_df['corrn'], clean_df['gOSI'])
        if p < 0.05:
            if corr > 0:
                summary.append("- Neurons with higher orientation selectivity (gOSI) tend to have stronger RNN correlations, suggesting that the RNN model more effectively captures the responses of orientation-selective neurons.")
            else:
                summary.append("- Neurons with higher orientation selectivity (gOSI) tend to have weaker RNN correlations, suggesting that the RNN model may struggle to capture the more specialized response patterns of highly orientation-selective neurons.")
        else:
            summary.append("- No significant overall relationship was found between orientation selectivity (gOSI) and RNN correlation, suggesting that the RNN's performance is not systematically affected by a neuron's orientation tuning strength.")
        
        session_corrs = [
            pearsonr(clean_df[clean_df['session'] == session]['corrn'], 
                    clean_df[clean_df['session'] == session]['gOSI'])[0]
            for session in clean_df['session'].unique()
        ]
        
        if max(session_corrs) > 0.2 and min(session_corrs) < -0.2:
            summary.append("- There is substantial variability in the relationship between RNN correlation and gOSI across sessions, suggesting that session-specific factors (e.g., recording quality, brain state) may significantly influence how well the RNN captures neural responses.")
        
        # Volume effects
        corr, p = pearsonr(clean_df['corrn'], clean_df['volume'])
        if p < 0.05:
            if corr > 0:
                summary.append("- Larger neurons tend to have stronger RNN correlations, possibly because they provide stronger, more reliable signals that are easier for the RNN to model.")
            else:
                summary.append("- Smaller neurons tend to have stronger RNN correlations, which could indicate that the RNN better captures the activity patterns of smaller, potentially more specialized neurons.")
        else:
            summary.append("- Neuron volume does not significantly predict RNN correlation, suggesting that physical size is not a major determinant of how well the RNN can model a neuron's responses.")
        
        # Feature importance interpretation
        if feature_importance_results:
            top_feature = feature_importance_results['feature_importance'].iloc[0]['Feature']
            summary.append(f"- {top_feature} was identified as the most important feature for predicting RNN correlation, indicating its particular relevance to the RNN's performance in modeling neural responses.")
        
        # Cluster analysis interpretation
        if cluster_results and 'anova_results' in cluster_results:
            if any(stats['p'] < 0.05 for stats in cluster_results['anova_results'].values()):
                summary.append("- Cluster analysis identified distinct subpopulations of neurons with significantly different characteristics, suggesting functional heterogeneity in how neurons relate to the RNN model.")
        
        # Write summary to file
        summary_text = '\n'.join(summary)
        summary_file = os.path.join(output_dir, 'summary_report.md')
        
        with open(summary_file, 'w') as f:
            f.write(summary_text)
        
        log(f"Summary report saved to {summary_file}")
        
        return summary_text
    
    # Generate summary report
    summary_report = generate_summary_report()
    
    # 7. Return Results
    # ===============
    log("Analysis complete. Results saved to {output_dir}")
    
    # Compile and return results
    results = {
        'dataset_stats': {
            'total_neurons': len(clean_df),
            'sessions': clean_df['session'].unique().tolist(),
            'scans': clean_df['scan'].nunique(),
            'overall_stats': overall_stats.to_dict(),
            'correlation_matrix': correlation_matrix.to_dict(),
            'session_stats': session_stats.to_dict(),
            'session_correlations': session_correlations
        },
        'regression_results': {
            'corrn_vs_gOSI': reg_corrn_gOSI,
            'corrn_vs_volume': reg_corrn_volume,
            'corrn_vs_orientation': reg_corrn_ori
        },
        'binned_analysis': {
            'corrn_gOSI': binned_corrn_gOSI,
            'gOSI_corrn': binned_gOSI_corrn,
            'volume_corrn': binned_volume_corrn
        },
        'feature_importance': feature_importance_results,
        'cluster_analysis': cluster_results,
        'bootstrap_results': {
            'corrn_gOSI': bootstrap_corrn_gOSI,
            'corrn_volume': bootstrap_corrn_volume
        },
        'output_directory': output_dir,
        'summary_report': summary_report
    }
    
    return results
comprehensive_neural_analysis()